# Lab 09 — Vector Databases & RAG

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- explain how **embeddings** turn meaning into geometry and why **cosine similarity** is the standard relevance score,
- implement **fixed-size chunking with overlap** and explain the chunk-size trade-off,
- build a complete plain-Python **RAG pipeline** — ingest → index → retrieve → generate — with NumPy as the "vector database",
- write a **grounded generation prompt** with chunk-ID citations and an explicit refusal path,
- evaluate retrieval with a labelled query set (**hit rate@k**, MRR) *before* judging generation,
- upgrade the fixed pipeline with a simple **assess-and-re-retrieve** step — retrieval as a tool the agent controls.

> ⏱️ Estimated time: 90–120 minutes. The corpus is tiny by design — every embedding and
> generation call runs locally via Ollama in seconds.

## Theory recap — finding five paragraphs in a million, and answering from them honestly

### Embeddings: meaning as geometry

An **embedding model** is a transformer encoder that maps a text span — a sentence, a paragraph,
a chunk — to a single dense vector (typically 256–3072 dimensions); the per-token representations
are **pooled** into one vector per span, so an embedding is a *summary*. Trained **contrastively**
(Reimers & Gurevych, 2019) — related pairs pulled together, unrelated pairs pushed apart — the
resulting geometry makes semantic similarity measurable as proximity. The standard score is
**cosine similarity**, $\cos\theta = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}$: it
compares *direction, not length*, and with L2-normalised vectors it reduces to a plain dot
product. Two lecture caveats: *similarity is not relevance* (embeddings are largely blind to
negation), and **retrieval asymmetry** — questions do not look like their answers, so good
retrieval models are trained on question–passage pairs, often with distinct instruction prefixes
for queries and documents.

### Searching the vectors

Exact nearest-neighbour search costs $O(N \cdot d)$ per query — a brute-force scan is fine at
ten thousand vectors and hopeless at a hundred million. At scale, **ANN** indexes such as
**HNSW** (layered proximity graphs, roughly logarithmic hops) trade a measured, tunable loss of
recall for orders of magnitude in speed. Our lab corpus is tiny, so we deliberately use the
exact NumPy scan — the honest choice at this scale; over-engineering is a real disease in this
field. The four vector-store categories (library, embedded DB, SQL extension, dedicated store)
all wrap similar indexes — the differences are operational.

### Chunking: how you split decides what can be found

Documents are chunked for three reasons: **sharp vectors** (one vector must summarise each
chunk), the **prompt budget**, and **citation granularity**. Strategies: fixed-size windows
with overlap (simple, format-agnostic, cuts thoughts mid-sentence), structural (split at the
author's joints), semantic (split where adjacent similarity drops), and contextual enrichment
(prepend document context to each chunk). The **chunk-size dial** has no universal optimum:
small chunks give sharp vectors and precise citations but strip context; large chunks are
self-contained but embed blurrily, drag in irrelevant text, and exhaust the budget.

### The classic RAG pipeline

**Offline:** ingest (load, clean, chunk) and index (embed chunks, store vectors). **Online:**
retrieve (embed the query *with the same model*, top-k by similarity), optionally rerank, then
generate (Lewis et al., 2020). **Grounding** is prompt discipline: system rules, then the
retrieved chunks with IDs, then the question; answer *only* from the sources, cite a chunk ID
per claim, and make refusal — "the sources do not say" — a first-class output. Position effects
apply: models privilege the start and end of long contexts (Liu et al., 2024).

### Evaluation and honest limits

Measure **retrieval before generation** — retrieval quality is the ceiling: a chunk that never
reaches the prompt cannot be cited by any generator. Against a labelled query set you measure
**recall@k** (in this lab: a per-query *hit rate*), **precision@k**, and rank-aware scores like
**MRR**. And the honest claim about hallucination: RAG *reduces* it and makes errors auditable
via citations — it does not eliminate it. Silent blending, retrieval failure with weak refusal,
and conflicting sources all survive; and a **faithful answer is still wrong when the corpus is
wrong** — RAG relocates trust from the model to the corpus.

## Part A — Setup

We keep dependencies minimal: `ollama`, `numpy`, `pandas`, `matplotlib`. Two local models are
used: a chat model for generation and `nomic-embed-text` for embeddings — the embedding model
is treated as a primitive, exactly as in the lecture.

In [ ]:
import json
import os
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ollama

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")           # chat / generation
EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "nomic-embed-text")  # embeddings

print(f"chat model:  {MODEL}")
print(f"embed model: {EMBED_MODEL}")

In [ ]:
# Connectivity check — run this before anything else.
try:
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "user", "content": "Reply with the single word: ready"}])
    print("chat model  OK ->", r["message"]["content"].strip()[:40])
    e = ollama.embed(model=EMBED_MODEL, input=["connectivity probe"])
    print(f"embed model OK -> {EMBED_MODEL} returns {len(e['embeddings'][0])}-dimensional vectors")
except Exception as exc:
    print("Ollama is not reachable or a model is missing:", exc)
    print("Fix: start Ollama with `ollama serve`, then pull the models with:")
    print("  ollama pull qwen2.5:7b")
    print("  ollama pull nomic-embed-text")

## Part B — Corpus and chunking

Our corpus is the **AURORA handbook**: six short technical documents about a *fictional*
research-agent platform (`data/*.md`) — its embedding service, vector store, ingestion
pipeline, retrieval API and incident log. Every fact in it is invented. That is the point:
the chat model cannot know these facts from pretraining, so **any correct answer must have
come from retrieval** — grounding becomes testable.

This is exactly the *ingest* stage of the pipeline: load, clean (our files are already clean),
chunk. We implement the lecture's baseline strategy — **fixed-size windows with overlap** —
approximating tokens by whitespace-separated words, which is good enough for a lab.

In [ ]:
DATA = Path("data") if Path("data").exists() else Path("../Lab09_RAG/data")

docs = []
for path in sorted(DATA.glob("*.md")):
    text = ___  # read the file's contents as one string
    docs.append({"doc": path.stem, "text": text})

print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(f"  {d['doc']:<22} {len(d['text'].split()):>4} words")

<details>
<summary><b>Click here for the solution</b></summary>

```python
DATA = Path("data") if Path("data").exists() else Path("../Lab09_RAG/data")

docs = []
for path in sorted(DATA.glob("*.md")):
    text = path.read_text(encoding="utf-8")  # read the file's contents as one string
    docs.append({"doc": path.stem, "text": text})

print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(f"  {d['doc']:<22} {len(d['text'].split()):>4} words")
```

</details>

In [ ]:
def chunk_text(text, chunk_size=80, overlap=20):
    """Fixed-size sliding windows over whitespace words, with overlap.

    `chunk_size` and `overlap` are counted in words (our token approximation).
    Consecutive windows share `overlap` words, so a sentence severed at a window
    boundary survives intact in the neighbouring chunk.
    """
    words = text.split()
    step = ___  # how far the window advances: chunk_size minus overlap
    chunks = []
    for start in range(0, len(words), step):
        window = ___  # the words from `start` up to `start + chunk_size`
        chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break  # this window already reached the end of the text
    return chunks


demo = chunk_text(docs[0]["text"], chunk_size=60, overlap=15)
print(f"{docs[0]['doc']}: {len(demo)} chunks of ~60 words (overlap 15)")
print("--- chunk 0 ends with:  ...", demo[0][-70:])
print("--- chunk 1 starts with:", demo[1][:70], "...")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    step = chunk_size - overlap  # how far the window advances
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]  # the words from `start` up to `start + chunk_size`
        chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break  # this window already reached the end of the text
    return chunks
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The window *advances* by `chunk_size - overlap` words, so consecutive windows share exactly
`overlap` words — that is the whole overlap trick. Example with `chunk_size=60, overlap=15`:
window 1 covers words 0–59, window 2 covers words 45–104. Compare the printed output: the end
of chunk 0 reappears at the start of chunk 1. The `break` prevents a final micro-chunk that
would only repeat the tail of the previous window: once a window has reached the end of the
text, we stop. Note what this baseline strategy does *not* know: where a heading, sentence or
table ends — that is the price of format-agnostic simplicity, and exactly what structural
chunking (used by AURORA's real ingestion pipeline, see `ingestion_pipeline.md`) improves on.

</details>

In [ ]:
def build_chunks(docs, chunk_size=80, overlap=20):
    """Chunk every document; give each chunk a citable ID like 'vector_store:2'."""
    records = []
    for d in docs:
        pieces = ___  # chunk this document's text with chunk_size and overlap
        for i, piece in enumerate(pieces):
            records.append({"chunk_id": f"{d['doc']}:{i}", "doc": d["doc"], "text": piece})
    return records


CHUNKS = build_chunks(docs, chunk_size=80, overlap=20)
print(f"Corpus -> {len(CHUNKS)} chunks (chunk_size=80, overlap=20)")
pd.DataFrame(CHUNKS).groupby("doc").size().rename("chunks per document").to_frame()

<details>
<summary><b>Click here for the solution</b></summary>

```python
def build_chunks(docs, chunk_size=80, overlap=20):
    records = []
    for d in docs:
        pieces = chunk_text(d["text"], chunk_size, overlap)  # chunk this document's text
        for i, piece in enumerate(pieces):
            records.append({"chunk_id": f"{d['doc']}:{i}", "doc": d["doc"], "text": piece})
    return records
```

The `chunk_id` is what the generator will later cite — *citation granularity* is one of the
three reasons to chunk at all.

</details>

> **Q:** Give the three reasons why documents are chunked rather than embedded whole.
<details><summary>Click for answer</summary>

First, **embedding sharpness**: one vector summarises its input, and a whole document averages
many topics into a blurry vector that matches queries poorly. Second, the **prompt budget**:
retrieved units must be small enough that several fit into the generation context. Third,
**citation granularity**: claims should be auditable to a paragraph, not to a 40-page document.

</details>

> **Q:** Explain the chunk-size trade-off in both directions.
<details><summary>Click for answer</summary>

**Small chunks (100–300 tokens):** sharp single-topic vectors, precise retrieval and citations —
but context is stripped away: dangling pronouns, dismembered tables and proofs, generator
confusion. **Large chunks (500–1500 tokens):** self-contained context for the generator — but
the embedding averages several topics into one blurry vector, every hit drags irrelevant text
into the prompt, and the budget is gone after a few hits. The optimum is corpus- and
query-dependent and must be found empirically — which is what Part G does.

</details>

## Part C — Embedding, indexing, retrieval

Now the *index* and *retrieve* stages. Our "vector database" is deliberately a **NumPy matrix
with an exact brute-force scan** — at a few dozen vectors this is the honest engineering choice
(the lecture: brute force is fine until it is not; AURORA's own store falls back to an exact
NumPy scan below 50,000 vectors). No FAISS, no Chroma — you build the primitive yourself first.

One lecture point becomes concrete here: **retrieval asymmetry**. `nomic-embed-text` is a
retrieval-trained model that expects distinct instruction prefixes — `search_document:` for
indexed texts and `search_query:` for queries — so that questions and answer passages land in
compatible regions of the space. And we embed index *and* query with the **same model**: a
version mismatch between the two paths returns silent noise (the corpus documents exactly this
incident, QF-5501).

In [ ]:
DOC_PREFIX = "search_document: "   # for texts that go INTO the index
QUERY_PREFIX = "search_query: "    # for the questions we search WITH


def embed_texts(texts, prefix):
    """Embed a list of strings -> (n, d) float32 NumPy matrix with L2-normalised rows."""
    try:
        resp = ___  # embed the prefixed texts with EMBED_MODEL (hint: ollama.embed)
    except Exception as exc:
        raise RuntimeError(
            "Embedding call failed — is Ollama running (`ollama serve`) and "
            "`nomic-embed-text` pulled (`ollama pull nomic-embed-text`)?") from exc
    M = np.array(resp["embeddings"], dtype=np.float32)
    norms = ___  # L2 norm of every row, shape (n, 1) (hint: np.linalg.norm with keepdims)
    return M / norms


probe = embed_texts(["cosine similarity compares direction, not length"], DOC_PREFIX)
print("shape:", probe.shape, "| row norm:", round(float(np.linalg.norm(probe[0])), 4))

<details>
<summary><b>Click here for the solution</b></summary>

```python
def embed_texts(texts, prefix):
    try:
        resp = ollama.embed(model=EMBED_MODEL, input=[prefix + t for t in texts])
    except Exception as exc:
        raise RuntimeError(
            "Embedding call failed — is Ollama running (`ollama serve`) and "
            "`nomic-embed-text` pulled (`ollama pull nomic-embed-text`)?") from exc
    M = np.array(resp["embeddings"], dtype=np.float32)
    norms = np.linalg.norm(M, axis=1, keepdims=True)  # L2 norm of every row, shape (n, 1)
    return M / norms
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`ollama.embed` accepts a list and returns one embedding per input — one batched call instead
of a Python loop. We normalise every row to unit length (`keepdims=True` keeps the shape
`(n, 1)` so broadcasting divides each row by its own norm). After normalisation, cosine
similarity between any two vectors is just their dot product:
$\cos\theta = u \cdot v$ when $\lVert u \rVert = \lVert v \rVert = 1$. That turns the entire
retrieval step in the next cells into a single matrix–vector product.

</details>

In [ ]:
# The "vector database": one matrix, rows aligned with CHUNKS.
MATRIX = ___  # embed every chunk's text as a *document*
print(f"Index built: {MATRIX.shape[0]} vectors x {MATRIX.shape[1]} dimensions "
      f"({MATRIX.nbytes / 1e6:.2f} MB) — brute force is fine at this scale.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
MATRIX = embed_texts([c["text"] for c in CHUNKS], DOC_PREFIX)
print(f"Index built: {MATRIX.shape[0]} vectors x {MATRIX.shape[1]} dimensions "
      f"({MATRIX.nbytes / 1e6:.2f} MB) — brute force is fine at this scale.")
```

This is the *offline* phase of the pipeline: it runs once per corpus. Everything from the next
cell on is the *online* phase, once per query.

</details>

In [ ]:
def retrieve(query, k=3):
    """Top-k chunks by cosine similarity. Returns a list of (chunk_record, score).

    The score is returned, not hidden — the assess step in Part F needs it.
    """
    q = embed_texts([query], QUERY_PREFIX)[0]
    sims = ___  # cosine similarity of q to every row of MATRIX (one matrix-vector product)
    order = ___  # indices of the k largest similarities, best first (hint: np.argsort)
    return [(CHUNKS[i], float(sims[i])) for i in order]

<details>
<summary><b>Click here for the solution</b></summary>

```python
def retrieve(query, k=3):
    q = embed_texts([query], QUERY_PREFIX)[0]
    sims = MATRIX @ q  # cosine similarity of q to every row (rows and q are unit length)
    order = np.argsort(sims)[::-1][:k]  # indices of the k largest similarities, best first
    return [(CHUNKS[i], float(sims[i])) for i in order]
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`MATRIX @ q` computes all $N$ dot products at once — because every row and the query are
L2-normalised, these *are* the cosine similarities. This line is the exact $O(N \cdot d)$
scan from the lecture: at $N$ in the dozens it costs microseconds; at $N = 10^8$ under
interactive latency it is hopeless, which is where HNSW takes over. `np.argsort` sorts
ascending, so `[::-1]` reverses to descending and `[:k]` keeps the top k. A real system would
use `np.argpartition` for large $N$ — clarity wins here.

</details>

In [ ]:
sample_query = "How many dimensions do the embedding vectors have?"
hits = ___  # the top 3 chunks for sample_query

pd.DataFrame([{"chunk_id": rec["chunk_id"], "cosine": round(score, 3),
               "text": rec["text"][:90] + "..."} for rec, score in hits])

<details>
<summary><b>Click here for the solution</b></summary>

```python
sample_query = "How many dimensions do the embedding vectors have?"
hits = retrieve(sample_query, k=3)
```

The top hit should come from `embedding_service` and contain "768 dimensions" — note that the
question and the passage share almost no surface form ("How many…" vs "…mapped to a single
vector of 768 dimensions"). That is dense retrieval matching *meaning*, and the
retrieval-asymmetry training paying off.

</details>

> **Q:** Why is cosine similarity the standard comparison for embeddings rather than Euclidean distance?
<details><summary>Click for answer</summary>

Cosine similarity compares only the **direction** of two vectors, ignoring their lengths —
vector norms can vary with text length and token statistics without carrying semantic signal,
so length is a nuisance dimension. With L2-normalised vectors, cosine similarity reduces to a
dot product and is monotonically related to Euclidean distance, so the choice is also
computationally cheap and index-friendly.

</details>

> **Q (not exam-relevant):** Why does `nomic-embed-text` require the prefixes `search_document:` and `search_query:`, and what happens if you mix them up?
<details><summary>Click for answer</summary>

This is the lecture's **retrieval asymmetry** made concrete: a question and the passage that
answers it share little surface form, so retrieval-trained models learn *two* mappings — one
for queries, one for documents — selected by an instruction prefix, that place both sides in
compatible regions of the space. Mixing the prefixes up (or omitting them) raises no error;
retrieval quality just silently degrades — the classic silent failure mode: everything runs,
nothing matches well. The exam-relevant core is the asymmetry concept, not this model's
prefix strings.

</details>

> **Q:** Why does exact nearest-neighbour search become infeasible at scale — and when is brute force nevertheless the right choice?
<details><summary>Click for answer</summary>

Exact search is $O(N \cdot d)$ per query: every stored vector must be compared, so cost grows
linearly with collection size and query rate. At around a million vectors a vectorised scan
still answers in tens of milliseconds, so for small collections, low query rates, or one-off
batch jobs, brute force is simpler *and exactly correct* — our lab index is precisely this
case. ANN indexes like HNSW become necessary when $N$, the query rate, or latency budgets make
the linear scan the bottleneck; they return ~99% of the true neighbours for orders of
magnitude less work, a measured and tunable trade.

</details>

## Part D — Grounded generation with citations

The *generate* stage. Retrieval only pays off if the prompt enforces it — grounding is
**prompt discipline** (Session 04's craft applied to a new problem). The assembled prompt has
three parts, in this order: **system rules**, then the **retrieved chunks with IDs**, then the
**question**. The rules encode the lecture's three commandments:

1. answer **only** from the provided sources,
2. **cite the chunk ID** after every factual claim — the audit trail,
3. the **refusal path**: if the sources do not contain the answer, *say so* — never improvise.

We then test the pipeline on an **answerable** and an **unanswerable** question. The
unanswerable case is the more important test: without a refusal path, a model handed
irrelevant chunks improvises from parametric memory while *appearing* grounded — the worst
failure mode, fluent and falsely credentialed (the corpus documents it as incident INC-2440).

In [ ]:
SYSTEM_RULES = """You are the answering module of a research agent.
Answer ONLY from the provided sources.
After every factual claim, cite the supporting chunk id in square brackets, e.g. [vector_store:2].
If the sources do not contain the answer, reply exactly:
"The sources do not contain the answer to this question."
Never use knowledge that is not in the sources."""


def build_prompt(question, hits):
    """Assemble the grounded prompt: rules, then chunks with IDs, then the question."""
    blocks = []
    for rec, score in hits:
        blocks.append(___)  # one source block: "[<chunk_id>] <chunk text>"
    context = "\n\n".join(blocks)
    user = f"Sources:\n\n{context}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": ___}]  # the assembled sources-plus-question message


print(build_prompt("test?", retrieve("chunk length", k=1))[1]["content"][:300])

<details>
<summary><b>Click here for the solution</b></summary>

```python
def build_prompt(question, hits):
    blocks = []
    for rec, score in hits:
        blocks.append(f"[{rec['chunk_id']}] {rec['text']}")  # one source block
    context = "\n\n".join(blocks)
    user = f"Sources:\n\n{context}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": user}]
```

Each chunk is wrapped with a stable identifier the model can cite. The question comes *last* —
with the strongest chunks first, this respects the position effects from the lecture (models
privilege the start and end of long contexts).

</details>

In [ ]:
def rag_answer(question, k=3):
    """The full online path: retrieve -> augment -> generate."""
    hits = retrieve(question, k=k)
    messages = build_prompt(question, hits)
    try:
        resp = ___  # call ollama.chat with MODEL and the assembled messages
    except Exception as exc:
        raise RuntimeError("Chat call failed — is Ollama running (`ollama serve`) "
                           "and the model pulled (`ollama pull qwen2.5:7b`)?") from exc
    return resp["message"]["content"], hits


ans, hits = rag_answer("What caused error QF-5501 in the ingestion service?")
print(ans)
print("\nretrieved:", [(rec["chunk_id"], round(score, 3)) for rec, score in hits])

<details>
<summary><b>Click here for the solution</b></summary>

```python
def rag_answer(question, k=3):
    hits = retrieve(question, k=k)
    messages = build_prompt(question, hits)
    try:
        resp = ollama.chat(model=MODEL, messages=messages)
    except Exception as exc:
        raise RuntimeError("Chat call failed — is Ollama running (`ollama serve`) "
                           "and the model pulled (`ollama pull qwen2.5:7b`)?") from exc
    return resp["message"]["content"], hits
```

Expected behaviour: the answer names the **embedding model version mismatch** and cites
something like `[incident_log:0]` or `[incident_log:1]`. The cited chunk ID must *resolve* —
you can look it up in `CHUNKS` and check that it really supports the claim. That is the audit
trail: for the research agent, a claim without a resolvable citation does not ship.

</details>

In [ ]:
unanswerable_question = "Who is the lead engineer of the AURORA project?"

ans_un, hits_un = ___  # ask the pipeline the unanswerable question above
print(ans_un)
print("\nbest similarity among retrieved chunks:", round(hits_un[0][1], 3),
      " <- keep this number in mind for Part F")

<details>
<summary><b>Click here for the solution</b></summary>

```python
unanswerable_question = "Who is the lead engineer of the AURORA project?"
ans_un, hits_un = rag_answer(unanswerable_question)
```

Retrieval cannot refuse: it *always* returns the k nearest chunks, relevant or not — note that
the best similarity is noticeably lower than for the answerable query. The refusal must
therefore come from the **generation** step, and only because the system rules make "the
sources do not contain the answer" a first-class output. Try deleting the refusal line from
`SYSTEM_RULES` and re-running: with a weak or missing refusal path, many models improvise a
plausible name — fluent, confident, and pure invention (incident INC-2440 in the corpus is
exactly this regression).

</details>

> **Q:** Why is the refusal instruction ("say so if the sources do not contain the answer") so important, and what happens without it?
<details><summary>Click for answer</summary>

Without it, a model handed empty or off-topic context improvises an answer from parametric
memory while retaining the grounded system's authoritative framing — fluent, possibly
cited-looking, and unverifiable. That is *worse* than an ungrounded system, because it borrows
false credibility. The refusal path makes "not in the sources" a first-class output and is the
cheapest single hallucination defence in RAG.

</details>

> **📝 Report task R2:** In Part D your pipeline faced an unanswerable question, and the corpus itself documents what happens when this goes wrong (incident INC-2440). Explain in your report: (i) why the refusal instruction is the cheapest single hallucination defence in a RAG system and what failure mode appears without it; (ii) why even a perfectly **faithful** answer from your pipeline can still be wrong — where does RAG relocate trust, and what does that imply for the research agent's source vetting; (iii) the three hallucination failure modes that remain even in a well-built RAG system.
> *No solution is provided — include your answer in your lab report.*

## Part E — Evaluating retrieval

**Measure retrieval before generation** — retrieval quality is a hard ceiling: a chunk that
never reaches the prompt cannot be used or cited by any generator, however good. Most
"the model is hallucinating" complaints in RAG systems are retrieval failures wearing a
trench coat.

Retrieval evaluation needs **ground truth**: labelled queries with their relevant evidence.
`data/queries.json` provides 13 queries — 10 answerable (each labelled with the document and
the key phrase that answers it) and 3 unanswerable. Our metric is a per-query **hit rate@k**:
the share of answerable queries for which at least one top-k chunk comes from the gold
document *and* contains the gold phrase. With one relevant fact per query, this is exactly the
lecture's **recall@k** on our labelled set — small but curated beats large and sloppy.

In [ ]:
with open(DATA / "queries.json", encoding="utf-8") as f:
    QUERIES = ___  # parse the JSON file into a list of dicts

answerable = [q for q in QUERIES if q["answerable"]]
unanswerable = [q for q in QUERIES if not q["answerable"]]
print(f"{len(QUERIES)} queries: {len(answerable)} answerable, {len(unanswerable)} unanswerable")
pd.DataFrame(QUERIES).head(4)

<details>
<summary><b>Click here for the solution</b></summary>

```python
with open(DATA / "queries.json", encoding="utf-8") as f:
    QUERIES = json.load(f)  # parse the JSON file into a list of dicts
```

</details>

In [ ]:
def is_hit(query_rec, hits):
    """True if any retrieved chunk comes from the gold document AND contains the gold phrase."""
    for rec, score in hits:
        if rec["doc"] == query_rec["gold_doc"] and ___:  # the gold phrase occurs in the chunk text
            return True
    return False


def hit_rate_at_k(queries, k):
    """Share of queries whose relevant evidence made it into the top k — our recall@k."""
    scored = [is_hit(q, retrieve(q["question"], k=k)) for q in queries]
    return ___  # the mean of the boolean list `scored`

<details>
<summary><b>Click here for the solution</b></summary>

```python
def is_hit(query_rec, hits):
    for rec, score in hits:
        if rec["doc"] == query_rec["gold_doc"] and \
           query_rec["gold_phrase"].lower() in rec["text"].lower():
            return True
    return False


def hit_rate_at_k(queries, k):
    scored = [is_hit(q, retrieve(q["question"], k=k)) for q in queries]
    return float(np.mean(scored))
```

Labelling by *document + phrase* instead of by chunk ID makes the ground truth robust: chunk
IDs change whenever the chunking parameters change (Part G re-chunks constantly), but the fact
itself does not move.

</details>

In [ ]:
for k in (1, 3, 5):
    hr = ___  # hit rate of the answerable queries at this k
    print(f"hit rate@{k}: {hr:.2f}")

<details>
<summary><b>Click here for the solution</b></summary>

```python
for k in (1, 3, 5):
    hr = hit_rate_at_k(answerable, k)  # hit rate of the answerable queries at this k
    print(f"hit rate@{k}: {hr:.2f}")
```

Hit rate can only rise with $k$ — a larger top-k is a superset. Typical values here: high
(0.8–1.0) already at $k = 3$, because six topically separated documents are an easy corpus.
The interesting movements come in Part G, when the chunking changes underneath. If hit rate@5
were low, no prompt or model change downstream could save you — recall is the ceiling.

</details>

> **Q:** Why should retrieval be evaluated before generation in a RAG system?
<details><summary>Click for answer</summary>

Retrieval quality is a hard ceiling: a chunk that never reaches the prompt cannot be used or
cited by any generator. Measuring generation first confounds the two stages — most
hallucination complaints in RAG systems trace back to retrieval failures. Evaluating in
pipeline order localises the fault and directs effort to the highest-leverage fix, which is
usually chunking or hybrid search, not the model.

</details>

> **📝 Report task R3 (code):** Complete the cell below: implement **mean reciprocal rank (MRR)** over the answerable queries. MRR is rank-aware: it rewards putting the relevant chunk *first*, which matters because the generator does not weight all prompt positions equally ("lost in the middle"). In your report, state your MRR@5 and explain in two or three sentences what MRR captures that hit rate@5 does not.
> *No solution is provided — include your code, your MRR@5 and the explanation in your lab report.*

In [ ]:
def mrr(queries, k=5):
    """Mean reciprocal rank: mean of 1/rank of the FIRST relevant chunk (0 if none in top k)."""
    reciprocal_ranks = []
    for q in queries:
        hits = retrieve(q["question"], k=k)
        best = 0.0
        for rank, (hit_rec, score) in enumerate(hits, start=1):
            if ___:  # this chunk is relevant for q (same test as in is_hit)
                best = ___  # the reciprocal of this rank
                break
        reciprocal_ranks.append(best)
    return float(np.mean(reciprocal_ranks))


print(f"MRR@5 (answerable queries): {mrr(answerable, k=5):.2f}")

## Part F — Assess and re-retrieve: the agentic upgrade

The classic pipeline is **single-shot**: one retrieval, one answer — nobody checks whether the
evidence was actually sufficient. Agentic RAG makes retrieval a **tool in the loop**:
*plan → rewrite → retrieve → assess → answer*, looping back on an evidence gap. The full
version is Self-RAG (Asai et al., 2023) with trained reflection tokens; the lecture's
pragmatic recipe, which you implement here, needs no training:

- **assess**: look at the best similarity score — `retrieve` returns it instead of hiding it,
  precisely so this step has a signal to read;
- **rewrite**: if the score is weak, have the LLM turn the question into an index-friendly
  keyword query (jargon in, colloquialisms out);
- **re-retrieve** once and keep the better hit set — *once*, because each loop iteration costs
  a model call plus a search, and an unbounded loop is an agent that searches forever
  (Session 02: stop conditions).

> **Q:** What failure modes of single-shot RAG does agentic RAG address, and by what mechanism?
<details><summary>Click for answer</summary>

Three: (1) **poor user phrasing** as a search query — fixed by query rewriting, the agent
crafts index-appropriate queries; (2) answers needing facts from **multiple retrieval steps** —
fixed by multi-hop, where hop one's answer parameterises hop two's query; (3) **silently
insufficient retrieval** — fixed by an explicit assess step that inspects the results and
triggers re-retrieval. The mechanism is structural: retrieval becomes a tool inside the agent
loop, called as often as the agent decides — bounded, because each iteration costs tokens and
latency.

</details>

> **📝 Report task R4 (code):** Complete the cell below — the simple **assess-and-re-retrieve** loop the lecture announced for this lab (the pragmatic version of Self-RAG's reflect-and-re-retrieve idea). Assess the retrieval: if the best cosine similarity is below `ASSESS_THRESHOLD`, have the LLM rewrite the question into a keyword-style search query, retrieve again, and keep the better hit set. In your report: justify your threshold choice from the similarities you observed for answerable vs unanswerable/vague queries, show one example where the rewrite changed the retrieved set, and state what the loop costs (extra calls) and why we bound it to one round.
> *No solution is provided — include your code and the justification in your lab report.*

In [ ]:
ASSESS_THRESHOLD = 0.55  # below this best-hit similarity we distrust the retrieval

REWRITE_PROMPT = ("Rewrite the following question as a short keyword-style search query for "
                  "a technical documentation index. Reply with the query only.\n\n"
                  "Question: {question}")


def assess_and_answer(question, k=3):
    """One assess-and-re-retrieve round, then a grounded answer."""
    hits = retrieve(question, k=k)
    best = hits[0][1]
    if best < ASSESS_THRESHOLD:                      # ASSESS: is the evidence weak?
        resp = ollama.chat(model=MODEL, messages=[
            {"role": "user", "content": REWRITE_PROMPT.format(question=question)}])
        new_query = ___  # the rewritten query text from the LLM response
        new_hits = ___  # retrieve again with the rewritten query
        print(f"[assess] weak retrieval (best {best:.2f}) -> rewrote to: {new_query!r} "
              f"(new best {new_hits[0][1]:.2f})")
        if new_hits[0][1] > best:                    # keep whichever set is stronger
            hits = new_hits
    answer = ollama.chat(model=MODEL, messages=build_prompt(question, hits))
    return answer["message"]["content"], hits


vague = "that thing where deleted stuff secretly sticks around in the index"
ans_v, hits_v = assess_and_answer(vague)
print("\n" + ans_v)
print("\nretrieved:", [(rec["chunk_id"], round(score, 3)) for rec, score in hits_v])

## Part G — Tuning: the chunk-size dial, k, and overlap

*(No gaps in this part — run, vary, observe.)*

This is the experiment the lecture announced for this lab: **sweep the chunk size on a fixed
query set and watch the trade-off in your own data**. We rebuild the index for several chunk
sizes and overlaps (cheap on this corpus), compute the hit rate per $(chunk\ size, k)$, and
then look qualitatively at what a "hit" actually feeds the generator at each setting — the
point is to *see* the trade-off, not to find a magic number.

Things to try beyond the defaults: extreme overlaps (0 vs half the chunk size), $k = 10$
(what does that do to the prompt in Part D?), and your own queries against `gold_rank`.

In [ ]:
SWEEP_CHUNK_SIZES = [40, 80, 160, 320]   # words per chunk
SWEEP_KS = [1, 3, 5]

INDEX_CACHE = {}


def index_for(chunk_size, overlap):
    """Build (or reuse) chunks + embedding matrix for a chunking setting."""
    key = (chunk_size, overlap)
    if key not in INDEX_CACHE:
        chunks = build_chunks(docs, chunk_size, overlap)
        matrix = embed_texts([c["text"] for c in chunks], DOC_PREFIX)
        INDEX_CACHE[key] = (chunks, matrix)
    return INDEX_CACHE[key]


def gold_rank(chunks, matrix, query_rec, k_max=5):
    """Rank (1-based) of the first relevant chunk in the top k_max, or None if absent."""
    q = embed_texts([query_rec["question"]], QUERY_PREFIX)[0]
    order = np.argsort(matrix @ q)[::-1][:k_max]
    for rank, i in enumerate(order, start=1):
        rec = chunks[i]
        if (rec["doc"] == query_rec["gold_doc"]
                and query_rec["gold_phrase"].lower() in rec["text"].lower()):
            return rank
    return None


rows = []
for cs in SWEEP_CHUNK_SIZES:
    chunks_cs, matrix_cs = index_for(cs, overlap=cs // 4)   # overlap fixed at 25%
    ranks = [gold_rank(chunks_cs, matrix_cs, q, k_max=max(SWEEP_KS)) for q in answerable]
    for k in SWEEP_KS:
        hr = float(np.mean([(r is not None and r <= k) for r in ranks]))
        rows.append({"chunk_size": cs, "n_chunks": len(chunks_cs), "k": k, "hit_rate": hr})

sweep = pd.DataFrame(rows)
sweep.pivot(index=["chunk_size", "n_chunks"], columns="k", values="hit_rate")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k in SWEEP_KS:
    sub = sweep[sweep["k"] == k]
    ax.plot(sub["chunk_size"], sub["hit_rate"], marker="o", label=f"k = {k}")
ax.set_xscale("log")
ax.set_xticks(SWEEP_CHUNK_SIZES)
ax.set_xticklabels(SWEEP_CHUNK_SIZES)
ax.set_xlabel("chunk size (words)")
ax.set_ylabel("hit rate@k")
ax.set_ylim(0, 1.05)
ax.set_title("Retrieval hit rate vs chunk size (overlap = 25% of chunk size)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# Overlap sweep at fixed chunk_size=80: overlap protects facts that straddle a window boundary.
rows_o = []
for ov in [0, 10, 20, 40]:
    chunks_o, matrix_o = index_for(80, overlap=ov)
    ranks = [gold_rank(chunks_o, matrix_o, q, k_max=3) for q in answerable]
    rows_o.append({"overlap": ov, "n_chunks": len(chunks_o),
                   "hit_rate@3": float(np.mean([(r is not None and r <= 3) for r in ranks]))})
pd.DataFrame(rows_o)

In [ ]:
# The other side of the trade-off: WHAT does a hit feed the generator?
probe_q = "What caused error QF-5501 in the ingestion service?"
for cs in (40, 320):
    chunks_c, matrix_c = index_for(cs, overlap=cs // 4)
    qv = embed_texts([probe_q], QUERY_PREFIX)[0]
    top = int(np.argsort(matrix_c @ qv)[::-1][0])
    n_words = len(chunks_c[top]["text"].split())
    print(f"--- chunk_size {cs}: top-1 = {chunks_c[top]['chunk_id']} "
          f"({n_words} words enter the prompt per hit)")
    print(textwrap.shorten(chunks_c[top]["text"], width=280), "\n")

# Small chunks: passage-level citations, sharp — but the fact may arrive without its heading
# or split across a boundary. Large chunks: the whole document rides in with every hit —
# self-contained, but the prompt budget fills after a few hits, precision per token drops,
# and the citation points at a document, not a passage. Hit rate alone does not show this;
# that is why recall and answer quality can move in opposite directions.

In [ ]:
# Optional interactive exploration (falls back gracefully without ipywidgets).
try:
    import ipywidgets as widgets

    @widgets.interact(chunk_size=SWEEP_CHUNK_SIZES, k=(1, 5))
    def explore(chunk_size=80, k=3):
        chunks_i, matrix_i = index_for(chunk_size, overlap=chunk_size // 4)
        ranks = [gold_rank(chunks_i, matrix_i, q, k_max=k) for q in answerable]
        hr = float(np.mean([(r is not None and r <= k) for r in ranks]))
        print(f"{len(chunks_i):>3} chunks | hit rate@{k} = {hr:.2f}")
except ImportError:
    print("ipywidgets not installed — rerun the sweep cells above with other values instead.")

> **📝 Report task R1:** Read your Part G sweep: how does hit rate@k move with chunk size and with k on this corpus — and where does the trade-off bite in the *other* direction, i.e. what do large chunks cost the **generator** even where the hit rate looks fine (see the qualitative comparison cell)? Argue with the lecture's terms — sharp vs blurry vectors, self-contained context, prompt budget, precision as prompt hygiene — and recommend one operating point (chunk size, overlap, k) for this corpus. Include the plot in your report.
> *No solution is provided — include your answer and the plot in your lab report.*

## Wrap-up

**What you built:** the complete retrieval layer of the research agent, in plain Python —
fixed-size chunking with overlap, an embedding index (a NumPy matrix plus one honest
brute-force scan), cosine top-k retrieval that *returns its scores*, grounded generation with
chunk-ID citations and a refusal path, retrieval evaluation against a curated query set, and
the assess-and-re-retrieve loop that turns the fixed pipeline into a tool the agent controls.

**Takeaways**

- Chunking is the highest-leverage decision in a RAG system — and the chunk-size dial has no
  universal optimum, only a trade-off you position per corpus (you plotted it).
- Same embedding model (and prefix discipline) for index and query — mismatches fail silently.
- Grounding is prompt discipline: sources with IDs, citations per claim, refusal as a
  first-class output.
- Evaluate retrieval first: recall@k is the ceiling on everything downstream.
- RAG *reduces* hallucination and makes errors auditable; it does not eliminate them —
  faithful is not correct, trust moves to the corpus.

**Next week (Session 10):** the agent can now act on retrieved, *untrusted* content — which
makes retrieval an attack channel. Unit 7 begins: sandboxing and guardrails — how to contain
what an agent can do when something, or someone, misleads it.

---

### 📝 For your lab report

| Task | Where | What to hand in |
|------|-------|-----------------|
| **R1** | Part G | Chunk-size/k analysis with plot + recommended operating point, argued with the lecture's trade-off terms |
| **R2** | Part D | Refusal path, "faithful is not correct" / trust relocation, and the three residual hallucination failure modes |
| **R3** | Part E | Completed MRR cell, your MRR@5, and what MRR captures that hit rate@5 does not |
| **R4** | Part F | Completed assess-and-re-retrieve cell, threshold justification from observed similarities, one rewrite example, loop cost |